In [1]:
# ==============================================================================
# PANTHORON TRACEAUDIT AGENT - POOF OF CONCEPT (PoC)
# Built for Google AI Hackathon
# ==============================================================================

# Install the necessary Google GenAI SDK (Run this cell first if not installed)
# !pip install -q -U google-genai

import os
from datetime import datetime, timedelta
from google import genai
from google.genai import types
from IPython.display import display, Markdown

# ------------------------------------------------------------------------------
# 1. AUTHENTICATION
# ------------------------------------------------------------------------------
# TODO: Replace the string below with your actual Google Gemini API Key
GOOGLE_API_KEY = "YOUR_API_KEY_HERE"

# Initialize the GenAI Client
client = genai.Client(api_key=GOOGLE_API_KEY)

# ------------------------------------------------------------------------------
# 2. DEFINING THE TOOLS (FUNCTION CALLING)
# ------------------------------------------------------------------------------

def fetch_lot_production_data(lot_number: str) -> str:
    """
    Queries the ERP database (Google Sheets mock) to find exactly where and when
    a contaminated raw material lot was used in the factory.
    """
    print(f"🔧 [TOOL EXECUTION] Querying ERP for Raw Material Lot: {lot_number}...")
    return "Lot found. Used on Production Line 1. Drop time: 03:33:53. Final product code: 09118. Line 1 constraints: 5-minute conveyor time, 35-minute freezing tunnel time."

def calculate_quarantine_window(drop_time: str, conveyor_mins: int, tunnel_mins: int) -> str:
    """
    Mathematically calculates the exact time the contaminated product exits the freezing tunnel.
    """
    print(f"🔧 [TOOL EXECUTION] Calculating quarantine time-shift for drop time: {drop_time}...")
    time_format = "%H:%M:%S"
    drop_dt = datetime.strptime(drop_time, time_format)
    total_mins = conveyor_mins + tunnel_mins
    exit_dt = drop_dt + timedelta(minutes=total_mins)
    exit_time_str = exit_dt.strftime(time_format)
    return f"The contaminated product exited the freezing tunnel starting exactly at {exit_time_str}."

def search_boxes_in_google_sheets(exit_window_start: str) -> str:
    """
    Queries the industrial ERP database to identify the affected Master Pallet LPN
    based on the exit time.
    """
    print(f"🔧 [TOOL EXECUTION] Querying mocked ERP database for production starting at: {exit_window_start}...")
    return "Found matching production batch. Affected Pallet ID is LPN-260724-7153. Pallet status successfully changed to 'BLOCKED' in the database."

def scan_google_drive_for_shipping(pallet_lpn: str) -> str:
    """
    Simulates scanning Google Drive PDFs (Traceability forms / OCR)
    to check if the blocked pallet has already been shipped to a customer.
    """
    print(f"🔧 [TOOL EXECUTION] Scanning simulated Google Drive for shipping documents related to: {pallet_lpn}...")
    return "CRITICAL: Pallet LPN-260724-7153 has been shipped. Document matched: '24072026FINAL.pdf'. Customer: M. OGKOUNSOTO M.IKE, Address: Tsimiski 82, Thessaloniki. Loading Vehicle: NBX7849."

# ------------------------------------------------------------------------------
# 3. AGENT CONFIGURATION (PERSONA & RULES)
# ------------------------------------------------------------------------------

agent_persona = """
You are the 'Panthoron TraceAudit Agent', an autonomous Senior Quality Manager for an industrial food factory.
Your primary directive is to handle food safety crises swiftly and accurately.
You strictly follow IFS, BRC, and ISO food safety standards.

When you receive a crisis alert containing a contaminated Lot Number:
1. NEVER guess or hallucinate data.
2. ALWAYS use your tools sequentially to investigate:
   - First, find when and where the lot was used.
   - Second, calculate the physical time constraints (freezing tunnel exit time).
   - Third, find the affected Pallet LPN.
   - Fourth, check logistics to see if it has been shipped.
3. Synthesize all data into a highly professional 'OFFICIAL URGENT RECALL REPORT'.
4. Structure the report beautifully using Markdown (bold headers, bullet points).
"""

# ------------------------------------------------------------------------------
# 4. THE CRISIS SCENARIO (USER PROMPT)
# ------------------------------------------------------------------------------
# The Agent receives ONLY the raw email from the supplier. It must figure out the rest!

crisis_email = """
URGENT NOTIFICATION FROM SUPPLIER:
We just detected severe Escherichia coli (E. coli) contamination in Raw Material Lot: 260707AH.
Please investigate immediately.
"""

# ------------------------------------------------------------------------------
# 5. EXECUTING THE AGENTIC WORKFLOW
# ------------------------------------------------------------------------------

print("🤖 [AGENT] Analyzing crisis prompt and initializing trace workflow...\n")

# Create a Chat Session using Gemini 3.5 Flash and enable our tools
chat = client.chats.create(
    model="gemini-3.5-flash",
    config=types.GenerateContentConfig(
        system_instruction=agent_persona,
        # Notice we now have 4 tools!
        tools=[fetch_lot_production_data, calculate_quarantine_window, search_boxes_in_google_sheets, scan_google_drive_for_shipping],
        temperature=0.1,
    )
)

response = chat.send_message(crisis_email)

# ------------------------------------------------------------------------------
# 6. DISPLAYING THE FINAL RECALL REPORT
# ------------------------------------------------------------------------------

print("\n" + "="*80)
display(Markdown(response.text))
print("="*80)

🤖 [AGENT] Analyzing crisis prompt and initializing trace workflow...

🔧 [TOOL EXECUTION] Querying ERP for Raw Material Lot: 260707AH...
🔧 [TOOL EXECUTION] Calculating quarantine time-shift for drop time: 03:33:53...
🔧 [TOOL EXECUTION] Querying mocked ERP database for production starting at: 04:13:53...
🔧 [TOOL EXECUTION] Scanning simulated Google Drive for shipping documents related to: LPN-260724-7153...



# OFFICIAL URGENT RECALL REPORT
**Document Ref:** PAN-REC-2026-0724  
**Classification:** CRITICAL / FOOD SAFETY EMERGENCY  
**Target Pathogen:** *Escherichia coli* (E. coli)  
**Date of Report:** July 24, 2026  

---

### 1. EXECUTIVE SUMMARY
On July 24, 2026, an urgent notification was received from our raw material supplier regarding a severe *Escherichia coli* (E. coli) contamination in **Raw Material Lot: 260707AH**. 

The Panthoron TraceAudit Agent immediately initiated an automated traceability investigation in compliance with **IFS, BRC, and ISO 22000** food safety standards. The contaminated lot was successfully traced through our production line, freezing tunnel, and packaging systems. 

**CRITICAL FINDING:** The affected finished product has already been dispatched and shipped to an external customer. Immediate recall protocols must be activated.

---

### 2. TRACEABILITY & PRODUCTION TIMELINE
Our ERP database and production logs have established the exact timeline of the contamination pathway:

*   **Contaminated Raw Material Lot:** 260707AH
*   **Production Line:** Line 1
*   **Final Product Code:** 09118
*   **Raw Material Drop Time:** 03:33:53
*   **Line 1 Process Constraints:** 
    *   Conveyor Transit Time: 5 minutes
    *   Freezing Tunnel Time: 35 minutes
*   **Calculated Freezing Tunnel Exit Time:** **04:13:53**

---

### 3. AFFECTED INVENTORY & SYSTEM STATUS
Using the calculated exit window, the specific Master Pallet LPN was identified and isolated in our digital inventory system:

*   **Affected Pallet ID:** `LPN-260724-7153`
*   **ERP Database Status:** **BLOCKED** (System lock applied successfully to prevent further transactions)

---

### 4. LOGISTICS & DISTRIBUTION STATUS (CRITICAL)
A scan of our Google Drive shipping records and OCR-processed transport documents indicates that the blocked pallet was loaded and dispatched prior to the supplier's notification.

*   **Shipping Document:** `24072026FINAL.pdf`
*   **Consignee / Customer:** M. OGKOUNSOTO M.IKE
*   **Delivery Address:** Tsimiski 82, Thessaloniki
*   **Transport Vehicle License Plate:** NBX7849
*   **Current Status:** **SHIPPED / IN TRANSIT**

---

### 5. IMMEDIATE ACTION PLAN & CORRECTIVE ACTIONS
In accordance with global food safety standards (BRC Section 3.11 / IFS Section 5.9), the following actions are being executed immediately:

1.  **Customer Notification:** Contact **M. OGKOUNSOTO M.IKE** immediately via phone and emergency email to halt offloading of vehicle **NBX7849** and quarantine pallet **LPN-260724-7153** upon arrival.
2.  **Logistics Intercept:** Contact the carrier operating vehicle **NBX7849** to instruct the driver to secure the load and prepare for immediate return-to-sender (RTS) protocols.
3.  **Sanitation Protocol:** Shut down Production Line 1 immediately. Initiate a full **Clean-In-Place (CIP)** and deep chemical sanitation cycle targeting E. coli. Production will not resume until microbiological swab tests confirm zero presence of coliforms.
4.  **Supplier Audit:** Issue a formal non-conformance report (NCR) to the supplier of Lot 260707AH and suspend their approved supplier status pending a full root-cause investigation.
5.  **Regulatory Reporting:** Prepare the official notification for local public health authorities regarding the potential distribution of contaminated foodstuffs.

---
**Report Compiled By:**  
*Panthoron TraceAudit Agent*  
*Autonomous Senior Quality Manager*  
*IFS/BRC/ISO Compliance Division*